# qrace quickstart — a reality-check on one European option

This notebook asks the only question `qrace` exists to answer:

> **Would a quantum computer actually help price _this_ option — and what would it cost?**

We take a vanilla European call, set a target accuracy, pick a noise profile, and let the
advisor return a structured `Verdict`. No hype: the answer for present-day noise levels is
usually **no**, and the verdict shows exactly why.

In [ ]:
from qrace import AnalysisTarget, NoiseProfile, OptionSpec, QiskitBackend, analyze

option = OptionSpec(
    kind="european_call", spot=100.0, strike=105.0, maturity=1.0, volatility=0.2, rate=0.05
)
target = AnalysisTarget(target_abs_error=0.5, confidence=0.95)
backend = QiskitBackend(seed=7)

## 1. The ideal-simulator baseline

First, a noise-free run: this checks the quantum pipeline itself (log-normal loading,
payoff encoding, iterative amplitude estimation) against the FinancePy analytic price.

In [ ]:
ideal = analyze(option, target, backend, NoiseProfile(kind="ideal"))
print(f"QAE estimate (ideal): {ideal.noise_aware_estimate:.4f} +/- {ideal.noise_aware_error:.4f}")
print(f"FinancePy reference:  {ideal.classical_reference:.4f}")
print(f"resources: {ideal.resource}")

The residual gap against the analytic price is *systematic*, not statistical: the
log-normal distribution is truncated and discretized onto a small grid, and the payoff is
encoded through a piecewise-linear approximation. The Shapley budget below attributes
each piece.

## 2. Now with realistic noise

A depolarizing error of 10⁻³ per two-qubit gate is *optimistic* for today's hardware.
Watch what it does.

In [ ]:
noisy = analyze(option, target, backend, NoiseProfile(kind="depolarizing", two_qubit_error=0.001))
print(f"QAE estimate (noisy): {noisy.noise_aware_estimate:.4f} +/- {noisy.noise_aware_error:.4f}")
print(f"FinancePy reference:  {noisy.classical_reference:.4f}")
print(f"required 2q fidelity: {noisy.required_two_qubit_fidelity:.6f}")
print(f"crossover: {noisy.crossover}")
print(f"quantum_advantageous: {noisy.quantum_advantageous}")

## 3. Where does the error budget go?

Exact Shapley attribution over the four pipeline stages. Efficiency holds by
construction: the contributions sum to the total deviation from the analytic truth.

In [ ]:
import matplotlib.pyplot as plt

contributions = noisy.shapley.contributions
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(list(contributions.keys()), list(contributions.values()))
ax.set_xlabel("Shapley contribution to |price error|")
ax.set_title(f"Error budget, total = {noisy.shapley.total:.3f}")
plt.tight_layout()
plt.show()

assert abs(sum(contributions.values()) - noisy.shapley.total) < 1e-9  # efficiency

## 4. Reading the verdict

For this option, at this accuracy, under optimistic noise:

- **decoherence dominates** the error budget — the circuit is too deep for the error rate;
- the **required two-qubit fidelity** (≈ 0.99998) exceeds any deployed hardware;
- at a loose ε = 0.5, classical Monte Carlo needs **fewer samples** than QAE needs oracle
  calls — the quadratic query advantage only bites at tight ε;
- so `quantum_advantageous = False`, with the reasons quantified rather than asserted.

See `02_report_and_shapley.ipynb` for grids across noise levels and the crossover curve.